In [ ]:
!apt-get update -qq && apt-get install -y -qq \
  build-essential python3-dev pkg-config \
  libosmesa6-dev patchelf libglew-dev libglfw3


In [ ]:
%%bash
set -e
cd /content
if [ ! -d PIToD/.git ]; then
  git clone https://github.com/NehaBhat14/PIToD.git PIToD
fi
cd PIToD
git pull


In [ ]:
# Mount Google Drive and symlink runs/ + figure/ so artifacts persist across sessions.
from google.colab import drive
import os
import shutil
import subprocess

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/PIToD_runs'
os.makedirs(f'{DRIVE_ROOT}/runs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/figure', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)

repo = '/content/PIToD'
for name in ('runs', 'figure'):
    local = os.path.join(repo, name)
    target = os.path.join(DRIVE_ROOT, name)
    if os.path.lexists(local):
        if os.path.islink(local) or os.path.isfile(local):
            os.unlink(local)
        else:
            shutil.rmtree(local)
    subprocess.run(['ln', '-sfn', target, local], check=True)

subprocess.run(['ls', '-la', os.path.join(repo, 'runs'), os.path.join(repo, 'figure')], check=False)


In [ ]:
%%bash
set -euo pipefail
MJ="${HOME}/.mujoco"
ARCH="${MJ}/mujoco210-linux-x86_64.tar.gz"
mkdir -p "${MJ}"
if [ ! -f "${MJ}/mujoco210/bin/libmujoco210.so" ]; then
  echo "Downloading MuJoCo 2.1.0 ..."
  wget -nv -O "${ARCH}" \
    "https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz" \
    || curl -fL -o "${ARCH}" \
    "https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz"
  SZ="$(stat -c%s "${ARCH}" 2>/dev/null || wc -c < "${ARCH}")"
  if [ "${SZ}" -lt 1000000 ]; then
    echo "Download looks too small (${SZ} bytes)."
    exit 1
  fi
  tar -xzf "${ARCH}" -C "${MJ}"
fi
test -f "${MJ}/mujoco210/bin/libmujoco210.so"
echo "MuJoCo OK at ${MJ}/mujoco210"


In [ ]:
%%bash
set -e
export MUJOCO_PY_MUJOCO_PATH="${HOME}/.mujoco/mujoco210"
export LD_LIBRARY_PATH="${MUJOCO_PY_MUJOCO_PATH}/bin:${LD_LIBRARY_PATH:-}"
pip install -r /content/PIToD/requirements-colab.txt


In [ ]:
%%bash
set -e
export MUJOCO_PY_MUJOCO_PATH="${HOME}/.mujoco/mujoco210"
export LD_LIBRARY_PATH="${MUJOCO_PY_MUJOCO_PATH}/bin:${LD_LIBRARY_PATH:-}"
pip install "mujoco-py==2.1.2.14" -v


In [ ]:
# Matched hyperparameters for static_pitod vs dynamic_pitod (see HOW_TO_RUN.md).
INFO = 'compare_static_vs_dynamic'
ENV = 'Hopper-v2'
SEED = 0
EPOCHS = 15
STEPS_PER_EPOCH = 5000
GPU_ID = 0

BASE_FLAGS = [
    '-env', ENV, '-seed', str(SEED),
    '-epochs', str(EPOCHS), '-steps_per_epoch', str(STEPS_PER_EPOCH),
    '-info', INFO, '-gpu_id', str(GPU_ID),
    '-layer_norm', '1', '-layer_norm_policy', '1',
    '-evaluate_bias', '1', '--experience_cleansing', '1',
    '-n_eval', '10', '--influence_estimation_interval', '5',
]

import os
import subprocess
import sys


def run_streaming(argv, log_name):
    """Run subprocess with live lines in this notebook cell + mirror log to Drive."""
    env = os.environ.copy()
    env.update({
        'PYTHONPATH': '/content/PIToD',
        'PYTHONUNBUFFERED': '1',
        'MUJOCO_PY_MUJOCO_PATH': f"{os.environ['HOME']}/.mujoco/mujoco210",
        'LD_LIBRARY_PATH': f"{os.environ['HOME']}/.mujoco/mujoco210/bin:"
        + env.get('LD_LIBRARY_PATH', ''),
        'MUJOCO_GL': 'osmesa',
    })
    log_path = f"/content/drive/MyDrive/PIToD_runs/logs/{log_name}.log"
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'w', encoding='utf-8', errors='replace') as logf:
        p = subprocess.Popen(
            argv,
            cwd='/content/PIToD',
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            text=True,
        )
        assert p.stdout is not None
        for line in p.stdout:
            print(line, end='', flush=True)
            logf.write(line)
            logf.flush()
        p.wait()
    print(f"\n[exit {p.returncode}] log -> {log_path}")
    return p.returncode


In [ ]:
# static_pitod = uniform replay + post-hoc PIToD logging (same script as dynamic).
rc = run_streaming(
    [sys.executable, '-u', 'dynamic-main-TH.py', *BASE_FLAGS, '--replay_mode', 'static_pitod'],
    log_name=f'{INFO}_static_pitod_s{SEED}',
)
assert rc == 0, f'static_pitod failed with exit {rc}'


In [ ]:
rc = run_streaming(
    [
        sys.executable,
        '-u',
        'dynamic-main-TH.py',
        *BASE_FLAGS,
        '--replay_mode',
        'dynamic_pitod',
        '--k_refresh',
        '5000',
        '--b_refresh',
        '32',
        '--dynamic_warmup_steps',
        '5000',
        '--m_strikes',
        '3',
        '--pitod_alpha',
        '0.6',
        '--n_samples_per_group',
        '64',
        '--h2_log',
        '1',
    ],
    log_name=f'{INFO}_dynamic_pitod_s{SEED}',
)
assert rc == 0, f'dynamic_pitod failed with exit {rc}'


In [ ]:
import bz2
import glob
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = f'/content/PIToD/runs/{INFO}'
runs_ = {}
for m in ('static_pitod', 'dynamic_pitod'):
    pat = f'{root}/redq_sac_{ENV}_{m}/redq_sac_{ENV}_{m}_s{SEED}'
    hits = sorted(glob.glob(pat))
    if not hits:
        raise FileNotFoundError(f'Missing run directory: {pat}')
    runs_[m] = hits[0]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
summary = {}
for m, d in runs_.items():
    df = pd.read_table(os.path.join(d, 'progress.txt'))
    ax[0].plot(df['Epoch'], df['AverageTestEpRet'], label=m)
    ax[1].plot(df['Epoch'], df['AverageQBias'], label=m)
    x = df['Epoch'].to_numpy()
    y = df['AverageTestEpRet'].to_numpy()
    if hasattr(np, 'trapezoid'):
        auc = float(np.trapezoid(y, x))
    else:
        auc = float(np.trapz(y, x))
    summary[m] = {
        'final': float(df['AverageTestEpRet'].iloc[-1]),
        'best': float(df['AverageTestEpRet'].max()),
        'auc': auc,
        'wallclock_s': float(df['Time'].iloc[-1]),
    }
ax[0].set_title('AverageTestEpRet')
ax[0].set_xlabel('epoch')
ax[0].legend()
ax[1].set_title('AverageQBias')
ax[1].set_xlabel('epoch')
ax[1].legend()
plt.tight_layout()
fig.savefig('/content/PIToD/figure/compare_learning.png', dpi=120)
plt.show()

print('\n== Summary (higher AUC/return = better for learning) ==')
for m, s in summary.items():
    print(
        f"{m:15s} final={s['final']:7.1f} best={s['best']:7.1f} "
        f"auc={s['auc']:10.1f} wallclock={s['wallclock_s']:.0f}s"
    )

fig2, ax2 = plt.subplots(1, 3, figsize=(15, 4))
cleansing_files = [
    'list_q_bias_cleansing.bz2',
    'list_q_bias_cleansing_valid.bz2',
    'list_return_cleansing.bz2',
]
interval = 5
for m, d in runs_.items():
    for j, fname in enumerate(cleansing_files):
        p = os.path.join(d, fname)
        if not os.path.isfile(p):
            continue
        with bz2.BZ2File(p, 'rb') as fh:
            arr = np.array(pickle.load(fh))[:, :, 0]
        x = np.arange(arr.shape[0]) * interval
        ax2[j].plot(x, arr[:, 0], '--', label=f'{m} pre')
        ax2[j].plot(x, arr[:, 1], '-', label=f'{m} post')
        ax2[j].set_title(fname.replace('.bz2', ''))
ax2[0].legend()
plt.tight_layout()
fig2.savefig('/content/PIToD/figure/compare_cleansing.png', dpi=120)
plt.show()
